# 🎬 Netflix Content Analysis
**Author:** Yamuna Saravanan  
**Tools:** Python (Pandas, Matplotlib, Seaborn)  
**Dataset:** Netflix Titles Dataset (8,807 rows × 12 columns)

---

## 📌 Project Overview
This project performs an end-to-end Exploratory Data Analysis (EDA) on Netflix's content library.
The goal is to uncover patterns in content type, genres, countries, ratings, and release trends
to understand how Netflix has grown and shifted its strategy over the years.

## 🎯 Key Questions Answered
1. What is the distribution of Movies vs TV Shows on Netflix?
2. Which genres, countries, and directors dominate the platform?
3. How has content addition grown over the years?
4. What are the most common content ratings?
5. Which months see the most content additions?

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries imported successfully')

---
## 2. Load Dataset

In [ ]:
data = pd.read_csv('netflix_titles.csv')
print(f'Dataset Shape: {data.shape}')
data.head()

---
## 3. Data Overview

In [ ]:
# Basic info
data.info()

In [ ]:
# Missing values summary
missing = data.isnull().sum()
missing_pct = (data.isnull().sum() / len(data) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing Values Summary:')
print(missing_df[missing_df['Missing Count'] > 0])

**Observation:** The `director` column has the highest missing values (~30%). `cast` and `country` also have notable gaps. `date_added` and `rating` have minor missing values. We will handle these in the cleaning step.

---
## 4. Data Cleaning
### 4.1 Data Type Conversion

In [ ]:
# Convert relevant object columns to category dtype
cat_cols = [i for i in data.columns 
            if data[i].dtype == 'O' 
            and i not in ('show_id','title','cast','date_added','duration',
                          'description','director','country','listed_in')]

data[cat_cols] = data[cat_cols].astype('category')
print(f'Converted to category: {cat_cols}')
print(data.dtypes)

### 4.2 Fix Rating Column (Misplaced Duration Values)

In [ ]:
# Check rating value counts — some duration values (e.g. '74 min') were entered in rating
print('Rating value counts:')
print(data['rating'].value_counts())

In [ ]:
# Fix: move misplaced duration values from rating → duration, replace rating with NR
mask = data['rating'].str.contains('min', na=False)
data.loc[mask & data['duration'].isnull(), 'duration'] = data.loc[mask & data['duration'].isnull(), 'rating']
data.loc[mask, 'rating'] = 'NR'
data['rating'].fillna('NR', inplace=True)
print('✅ Rating column cleaned')

### 4.3 Unnest Multi-Value Columns
Columns like `director`, `cast`, `country`, and `listed_in` contain comma-separated values. We unnest them so each value gets its own row — enabling accurate analysis.

In [ ]:
def unnest_column(df, col, index_col='title'):
    """Unnest a comma-separated column into individual rows."""
    exploded = df[col].apply(lambda x: str(x).split(', ')).tolist()
    new_df = pd.DataFrame(exploded, index=df[index_col])
    new_df = new_df.stack().reset_index(level=0)
    new_df = new_df.rename(columns={0: col})
    return new_df[[index_col, col]]

dfnew_director = unnest_column(data, 'director')
dfnew_cast     = unnest_column(data, 'cast')
dfnew_country  = unnest_column(data, 'country')
dfnew_genre    = unnest_column(data, 'listed_in')

print('✅ All multi-value columns unnested')
print(f'Director rows: {len(dfnew_director)} | Cast rows: {len(dfnew_cast)}')

In [ ]:
# Merge all unnested columns
merged = dfnew_director.merge(dfnew_cast, on='title', how='inner')
merged = merged.merge(dfnew_country, on='title', how='inner')
merged = merged.merge(dfnew_genre, on='title', how='inner')

# Replace nan strings
merged['director'].replace('nan', 'Unknown Director', inplace=True)
merged['cast'].replace('nan', 'Unknown Actor', inplace=True)
merged['country'].replace('nan', 'Unknown Country', inplace=True)

# Merge with core columns
core = data[['show_id','type','title','date_added','release_year','rating','duration','description']]
df_final = merged.merge(core, on='title', how='inner')
df_final.rename(columns={'listed_in': 'genre'}, inplace=True)

print(f'Final dataset shape: {df_final.shape}')
df_final.head(3)

### 4.4 Impute Missing Values

In [ ]:
# Impute date_added using mode of the same release_year group
for year in df_final[df_final['date_added'].isnull()]['release_year'].unique():
    mode_val = df_final[df_final['release_year'] == year]['date_added'].mode()
    if not mode_val.empty:
        df_final.loc[(df_final['release_year'] == year) & 
                     (df_final['date_added'].isnull()), 'date_added'] = mode_val[0]

# Impute country using director's most common country
for director in df_final[df_final['country'].isnull()]['director'].unique():
    if director in df_final[~df_final['country'].isnull()]['director'].unique():
        mode_country = df_final[df_final['director'] == director]['country'].mode()
        if not mode_country.empty:
            df_final.loc[(df_final['director'] == director) & 
                         (df_final['country'].isnull()), 'country'] = mode_country[0]

# Impute remaining country using cast's most common country
for actor in df_final[df_final['country'].isnull()]['cast'].unique():
    if actor in df_final[~df_final['country'].isnull()]['cast'].unique():
        mode_country = df_final[df_final['cast'] == actor]['country'].mode()
        if not mode_country.empty:
            df_final.loc[(df_final['cast'] == actor) & 
                         (df_final['country'].isnull()), 'country'] = mode_country[0]

# Fill remaining nulls
df_final['country'].fillna('Unknown Country', inplace=True)

print('✅ Missing values imputed')
print(df_final.isnull().sum())

### 4.5 Feature Engineering — Duration & Date

In [ ]:
# Convert date_added to datetime and extract month/year
df_final['date_added'] = pd.to_datetime(df_final['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df_final['month_added'] = df_final['date_added'].dt.month
df_final['year_added']  = df_final['date_added'].dt.year

# Duration: extract numeric minutes for movies
df_final['duration_clean'] = df_final['duration'].str.replace('min', '').str.strip()
df_final.loc[df_final['duration_clean'].str.contains('Season', na=False), 'duration_clean'] = 0
df_final['duration_clean'] = pd.to_numeric(df_final['duration_clean'], errors='coerce').fillna(0).astype(int)

# Bin duration into categories
bins  = [-1, 1, 50, 80, 100, 120, 150, 200, 315]
labels = ['TV Show','1–50','50–80','80–100','100–120','120–150','150–200','200–315']
df_final['duration_bin'] = pd.cut(df_final['duration_clean'], bins=bins, labels=labels)

print('✅ Feature engineering complete')
df_final.head(3)

---
## 5. Exploratory Data Analysis (EDA)
### 5.1 Movies vs TV Shows Distribution

In [ ]:
df_type = df_final.groupby('type')['title'].nunique().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
axes[0].pie(df_type['title'], labels=df_type['type'], autopct='%1.1f%%',
            explode=[0.05, 0], colors=['#E50914', '#221F1F'], startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('Movies vs TV Shows — Share', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(df_type['type'], df_type['title'], color=['#E50914', '#221F1F'], width=0.5)
axes[1].set_title('Movies vs TV Shows — Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Type')
axes[1].set_ylabel('Number of Titles')
for i, v in enumerate(df_type['title']):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('01_movies_vs_tvshows.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Netflix has nearly **3x more Movies than TV Shows**, but the platform has been increasingly investing in TV Show production post-2015.

### 5.2 Top 15 Genres

In [ ]:
df_genre = (df_final.groupby('genre')['title']
            .nunique()
            .reset_index()
            .sort_values('title', ascending=False)
            .head(15))

plt.figure(figsize=(12, 8))
colors = ['#E50914' if i == 0 else '#B81D24' if i < 3 else '#831010' for i in range(len(df_genre))]
plt.barh(df_genre['genre'][::-1], df_genre['title'][::-1], color=colors[::-1])
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Genre', fontsize=12)
plt.title('Top 15 Genres on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_top_genres.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** **International Movies, Dramas, and Comedies** are the top 3 genres — indicating Netflix's strong focus on global and emotionally engaging content.

### 5.3 Top 10 Content-Producing Countries

In [ ]:
df_country = (df_final[df_final['country'] != 'Unknown Country']
              .groupby('country')['title']
              .nunique()
              .reset_index()
              .sort_values('title', ascending=False)
              .head(10))

plt.figure(figsize=(12, 7))
sns.barplot(x='title', y='country', data=df_country, palette='Reds_r')
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.title('Top 10 Content-Producing Countries on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('03_top_countries.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** The **United States leads by a wide margin**, followed by **India and the UK**. India's strong position reflects Netflix's massive investment in regional content.

### 5.4 Top 10 Content Ratings

In [ ]:
df_rating = (df_final[df_final['rating'] != 'NR']
             .groupby('rating')['title']
             .nunique()
             .reset_index()
             .sort_values('title', ascending=False)
             .head(10))

plt.figure(figsize=(10, 6))
sns.barplot(x='title', y='rating', data=df_rating, palette='YlOrRd_r')
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Rating', fontsize=12)
plt.title('Top 10 Content Ratings on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('04_content_ratings.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** **TV-MA and TV-14** are the most common ratings, confirming Netflix primarily targets mature audiences.

### 5.5 Top 25 Actors by Appearance

In [ ]:
df_cast = (df_final[df_final['cast'] != 'Unknown Actor']
           .groupby('cast')['title']
           .nunique()
           .reset_index()
           .sort_values('title', ascending=False)
           .head(25))

plt.figure(figsize=(12, 10))
plt.barh(df_cast['cast'][::-1], df_cast['title'][::-1], color='#E50914')
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Actor', fontsize=12)
plt.title('Top 25 Most Appearing Actors on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05_top_actors.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.6 Top 10 Directors

In [ ]:
df_dir = (df_final[df_final['director'] != 'Unknown Director']
          .groupby('director')['title']
          .nunique()
          .reset_index()
          .sort_values('title', ascending=False)
          .head(10))

plt.figure(figsize=(12, 6))
sns.barplot(x='title', y='director', data=df_dir, palette='Reds_r')
plt.xlabel('Number of Titles', fontsize=12)
plt.ylabel('Director', fontsize=12)
plt.title('Top 10 Directors on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('06_top_directors.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.7 Most Experienced Directors (Active Years)

In [ ]:
df_exp = (df_final[df_final['director'] != 'Unknown Director']
          .groupby('director')['release_year']
          .agg(['min','max'])
          .reset_index())
df_exp['years_of_experience'] = df_exp['max'] - df_exp['min']
df_exp = df_exp.sort_values('years_of_experience', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='years_of_experience', y='director', data=df_exp, palette='YlOrRd_r')
plt.xlabel('Years of Experience on Netflix', fontsize=12)
plt.ylabel('Director', fontsize=12)
plt.title('Top 10 Most Experienced Directors on Netflix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('07_director_experience.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Trend Analysis
### 6.1 Content Added Per Year

In [ ]:
df_year = (df_final.groupby('year_added')['title']
           .nunique()
           .reset_index()
           .sort_values('year_added'))
df_year = df_year[df_year['year_added'] >= 2010]  # Focus on recent years

plt.figure(figsize=(12, 6))
plt.plot(df_year['year_added'], df_year['title'], marker='o', color='#E50914', linewidth=2.5, markersize=8)
plt.fill_between(df_year['year_added'], df_year['title'], alpha=0.15, color='#E50914')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Titles Added', fontsize=12)
plt.title('Netflix Content Added Per Year (2010–2021)', fontsize=14, fontweight='bold')
plt.xticks(df_year['year_added'], rotation=45)
plt.tight_layout()
plt.savefig('08_content_per_year.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Content additions **grew exponentially after 2016** and peaked around **2019**. The slight dip post-2019 aligns with COVID-19 production delays.

### 6.2 Movies vs TV Shows — Trend Over Years

In [ ]:
df_type_year = (df_final.groupby(['release_year','type'])['title']
                .nunique()
                .reset_index()
                .sort_values('release_year'))
df_type_year = df_type_year[df_type_year['release_year'] >= 2010]

plt.figure(figsize=(14, 7))
sns.barplot(x='release_year', y='title', hue='type', data=df_type_year,
            palette={'Movie':'#E50914','TV Show':'#221F1F'})
plt.xlabel('Release Year', fontsize=12)
plt.ylabel('Number of Titles', fontsize=12)
plt.title('Movies vs TV Shows Released Per Year (2010–2021)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.legend(title='Type')
plt.tight_layout()
plt.savefig('09_type_trend_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** **TV Show count grew sharply after 2015**, while Movie count started declining post-2018. This confirms Netflix's strategic shift toward serialized content to retain subscribers.

### 6.3 Content Additions by Month

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_month = (df_final.groupby('month_added')['title']
            .nunique()
            .reset_index()
            .sort_values('month_added'))
df_month['month_name'] = df_month['month_added'].apply(lambda x: month_labels[int(x)-1] if pd.notna(x) else 'Unknown')

plt.figure(figsize=(12, 6))
plt.bar(df_month['month_name'], df_month['title'], color='#E50914', edgecolor='white')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Number of Titles Added', fontsize=12)
plt.title('Netflix Content Additions by Month', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('10_content_by_month.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** **July and December** see the highest content additions — likely aligned with summer holidays and the year-end festive season.

### 6.4 Country-Wise Content: Movies vs TV Shows

In [ ]:
df_country_type = (df_final[df_final['country'] != 'Unknown Country']
                   .groupby(['country','type'])['title']
                   .nunique()
                   .reset_index()
                   .sort_values('title', ascending=False))

top_movie_countries = df_country_type[df_country_type['type']=='Movie'].head(10)
top_tv_countries    = df_country_type[df_country_type['type']=='TV Show'].head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.barplot(x='title', y='country', data=top_movie_countries, ax=axes[0], palette='Reds_r')
axes[0].set_title('Top 10 Movie-Producing Countries', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Movies')
axes[0].set_ylabel('Country')

sns.barplot(x='title', y='country', data=top_tv_countries, ax=axes[1], palette='Greys_r')
axes[1].set_title('Top 10 TV Show-Producing Countries', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of TV Shows')
axes[1].set_ylabel('')

plt.suptitle('Country-wise Content: Movies vs TV Shows', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('11_country_type_split.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** India ranks **2nd in Movies** but drops to **7th in TV Shows**, showing that Indian content is predominantly film-based. Japan and South Korea, however, lead in TV Show production.

---
## 7. Key Findings & Business Insights

| # | Insight |
|---|---|
| 1 | **Movies dominate** Netflix's library (75%) but **TV Shows are growing faster** post-2015 |
| 2 | **International Movies, Dramas & Comedies** are the most popular genres globally |
| 3 | **USA leads content production** by a large margin; India is 2nd, strong in Movies |
| 4 | **TV-MA & TV-14** are the top ratings, confirming Netflix targets adult audiences |
| 5 | Content additions **peaked in 2019** and slightly dipped post-2019 (COVID impact) |
| 6 | **July & December** are the busiest months for Netflix content additions |
| 7 | India ranks high in Movies but low in TV Shows — gap for local TV content growth |

---
## 8. Conclusion

This analysis reveals that Netflix has strategically evolved its content library over the years. 
While Movies still make up the majority of content, the platform's growing investment in **original TV Shows** 
post-2015 is a clear subscriber retention strategy. The dominance of **US, Indian, and UK content** highlights 
Netflix's focus on key global markets. Going forward, expanding **regional TV Show production** — especially 
in markets like India where Movies dominate — could be a key growth lever.

---
*Analysis by Yamuna Saravanan | GitHub: github.com/YamunaSk*